# DPRSC — C2 dimension-scaling (error vs dimension)

Empirically illustrates **Claim 2** of *Differentially Private Range Subgraph Counting* (ICML 2026, `QYpByrxSTg`): **any DP algorithm must incur additive error exponential in the dimension.**

We run the released DPRSC algorithms **and** the baselines at **d=1** and **d=2** and show the error grows ~exponentially with dimension. The theory is direct from the code: the Laplace noise magnitude is parameterized as `(ceil(log2(m))+1)^(2*d)`, so d=1 → d=2 should multiply error by ~`(ceil(log2(m))+1)^2` for every method (ours and baselines) — the 'exponential in dimension' behaviour C2 claims must hold for *any* DP algorithm.

**CPU-only** (numpy/pandas/matplotlib). **Fast config:** `edge` pattern only (enough for the dimension claim), d=1 at full query count, d=2 at 0.1x query count (n^2 is the slow part). ~2–4 min on free Colab. To also do `triangle`, add it to the pattern list (slower).

In [ ]:
# Clone the official repo (code + bundled datasets) and install deps
!pip -q install numpy pandas matplotlib
![ ! -d DPRSC ] && git clone --depth 1 https://github.com/Airleave/DPRSC
import os, sys
os.chdir('/content/DPRSC'); sys.path.insert(0, '/content/DPRSC')
print('ready at', os.getcwd())

In [ ]:
# Helper: epsilon (accuracy) test for one (pattern, dimension). Returns a DataFrame of
# MEAN relative error per epsilon for ours (pure_DP, approx_DP) and baselines (base_comp, base_comp_ADP).
import math, logging
import concurrent.futures
import numpy as np, pandas as pd
logging.basicConfig(level=logging.WARNING); logger = logging.getLogger('c2')
import preprocessing, ourAlg, baseline

def eps_run(dataset, n, pattern, d, eps_list, workers=2, qmult=1.0):
    d_max, test_nodes, edges, h, m = preprocessing.graph_data_load(dataset, pattern, n, logger, d)
    if d == 1:
        Qn = math.ceil(n**1.5 * qmult); Q = preprocessing.generate_queries(Qn, m, d)
    else:  # d == 2: upstream convention (Q_num = n^2, random queries)
        Qn = math.ceil(n**2 * qmult); Q = preprocessing.generate_random_queries(Qn, m, d)
    rows = []
    with concurrent.futures.ProcessPoolExecutor(max_workers=workers) as ex:
        ftrue = ex.submit(ourAlg.query_true, n, m, d, Q, test_nodes, logger)
        fp = {e: ex.submit(ourAlg.pure_DP, n, m, d, Q, e, test_nodes, pattern, logger) for e in eps_list}
        fa = {e: ex.submit(ourAlg.approx_DP, n, m, d, Q, d_max, e, 1e-5, test_nodes, pattern, logger) for e in eps_list}
        true = ftrue.result()
        for e in eps_list:
            pe, _ = fp[e].result(); ae, _ = fa[e].result()
            bc  = baseline.base_comp(n, Qn, 1, e, true, pattern, logger)
            bcA = baseline.base_comp_ADP(n, Qn, 1, d_max, e, 1e-5, true, pattern, logger)
            rows.append(dict(eps=e, pure=pe/Qn, approx=ae/Qn, base_comp=bc[0], base_comp_ADP=bcA[0]))
    return pd.DataFrame(rows), m
print('helper ready')

In [ ]:
# FAST RUN: edge pattern, d=1 (full Q) vs d=2 (0.1x Q). ~2-4 min on free Colab.
eps_list = [0.5, 1.0, 2.0, 4.0]
results = {}
for d, qm in [(1, 1.0), (2, 0.1)]:            # d=2 uses 10x fewer queries (n^2 is the slow part)
    print(f'>> edge d={d} (qmult={qm}) ...', flush=True)
    df, m = eps_run('ca-netscience', 379, 'edge', d, eps_list, workers=2, qmult=qm)
    results[('edge', d)] = (df, m)
    print(df.round(3).to_string(index=False), flush=True)
print('done')

In [ ]:
# d=2 / d=1 error ratio vs the theoretical noise factor (ceil(log2 m)+1)^2.
print('C2 — exponential-in-dimension check (d=2 error should be ~(log2 m+1)^2 x d=1 error):\n')
for pat in ['edge']:
    d1, _ = results[(pat, 1)]; d2, m = results[(pat, 2)]
    theo = (math.ceil(math.log2(m)) + 1) ** 2
    print(f'[{pat}] m={m}  ->  theoretical d2/d1 noise ratio = (log2(m)+1)^2 = {theo}')
    print(f"{'eps':>4} {'pure_d1':>9} {'pure_d2':>9} {'pure_ratio':>10} {'bc_d1':>8} {'bc_d2':>8} {'bc_ratio':>9}")
    for i in range(len(d1)):
        rp = d2.pure[i] / max(d1.pure[i], 1e-12)
        rb = d2.base_comp[i] / max(d1.base_comp[i], 1e-12)
        print(f'{d1.eps[i]:>4} {d1.pure[i]:>9.3g} {d2.pure[i]:>9.3g} {rp:>10.1f} {d1.base_comp[i]:>8.3g} {d2.base_comp[i]:>8.3g} {rb:>9.1f}')
    print()

In [ ]:
# Plot: mean error at d=1 vs d=2 for every method (log scale). Bars jump ~exponentially with d.
import matplotlib.pyplot as plt
pat = 'edge'
d1, _ = results[(pat, 1)]; d2, m = results[(pat, 2)]
methods = ['pure', 'approx', 'base_comp', 'base_comp_ADP']
m1 = [d1[k].mean() for k in methods]; m2 = [d2[k].mean() for k in methods]
fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(methods)); w = 0.38
ax.bar(x - w/2, m1, w, label='d=1'); ax.bar(x + w/2, m2, w, label='d=2')
ax.set_yscale('log'); ax.set_xticks(x); ax.set_xticklabels(methods, rotation=15)
ax.set_title(f'{pat} (m={m}): mean error d=1 vs d=2'); ax.set_ylabel('mean relative error (log)'); ax.legend()
plt.tight_layout(); plt.savefig('c2_dimension_scaling.png', dpi=110, bbox_inches='tight'); plt.show()
print('saved c2_dimension_scaling.png')

In [ ]:
# Save CSVs + zip everything for download (so the logbook can embed the numbers/figure).
import glob, zipfile
from google.colab import files
for (pat, d), (df, m) in results.items():
    df.assign(m=m).to_csv(f'c2_{pat}_d{d}.csv', index=False)
with zipfile.ZipFile('/content/dprsc_c2_results.zip', 'w') as zf:
    for f in glob.glob('c2_*'):
        zf.write(f)
files.download('/content/dprsc_c2_results.zip')
print('zipped', glob.glob('c2_*'))